# V4 Universal Football Model — Footballdata.io Ingestion

This notebook tests the new Footballdata.io API, replacing the deprecated Sofascore integration.
We will use this notebook to explore the JSON structure and build the parsing logic before wiring it into the live V4 backend.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load the API key from .env
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("FOOTBALLDATA_API_KEY")

if not API_KEY:
    print("⚠️ API Key not found! Please check your .env file.")
else:
    print(f"✅ API Key loaded: {API_KEY[:5]}...{API_KEY[-5:]}")

## 1. Basic API Fetch Function
Let's define a helper function to hit the Footballdata.io endpoints based on their documentation.

In [ ]:
# The correct base URL according to https://footballdata.io/documentation/endpoints/
BASE_URL = "https://footballdata.io/api/v1"

def fetch_footballdata(endpoint, params=None):
    """Helper to fetch data from footballdata.io"""
    if params is None:
        params = {}
        
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        if 'response' in locals() and response is not None:
            print(f"Response text: {response.text}")
        return None

## 2. Dynamic Fixture Parsing
The Footballdata `/fixtures/today` endpoint returns `data` as a dictionary (grouped by league or date) rather than a flat list. We will safely unpack it.

In [ ]:
print("Fetching /fixtures/today...")
today_data = fetch_footballdata("fixtures/today")
match_id_to_test = None

if today_data and today_data.get("success") and "data" in today_data:
    data_payload = today_data["data"]
    
    if isinstance(data_payload, list) and len(data_payload) > 0:
        match_id_to_test = data_payload[0].get("match_id")
    elif isinstance(data_payload, dict):
        for key, val in data_payload.items():
            if isinstance(val, list) and len(val) > 0:
                match_id_to_test = val[0].get("match_id")
                break
            elif isinstance(val, dict) and "matches" in val:
                match_list = val["matches"]
                if len(match_list) > 0:
                    match_id_to_test = match_list[0].get("match_id")
                    break

if match_id_to_test:
    print(f"\n✅ Found matches today! We will use match_id: {match_id_to_test} for detailed tests.")
else:
    print("\n⚠️ Could not automatically extract a match_id. Using the fallback match provided.")
    match_id_to_test = 780100645 # Manually testing the match you provided earlier

## 3. Deep Dive into Stats & Events
Let's fetch the granular match data so we can map `live_xg` and `red_cards`.

In [ ]:
if match_id_to_test:
    print(f"Fetching stats for match {match_id_to_test}...")
    stats_data = fetch_footballdata(f"matches/{match_id_to_test}/stats")
    
    print(f"Fetching events for match {match_id_to_test}...")
    events_data = fetch_footballdata(f"matches/{match_id_to_test}/events")


## 4. The V4 Ingestion Parser
Now that we know the structure (`data.events` is a list, `data.match.home_team` is a dict), we can build the parser. We also need to inspect the `stats` array to see how expected goals (xG) are stored.

In [ ]:
def parse_footballdata_to_v4(stats_payload, events_payload):
    """Parses Footballdata.io responses into the V4 standard schema."""
    match_info = stats_payload.get("data", {}).get("match", {})
    
    home_team = match_info.get("home_team", {}).get("team_name", "Unknown Home")
    away_team = match_info.get("away_team", {}).get("team_name", "Unknown Away")
    
    # 1. Parse Events (Goals, Cards)
    home_score = 0
    away_score = 0
    red_cards = {"home": 0, "away": 0}
    
    events = events_payload.get("data", {}).get("events", []) if events_payload else []
    for event in events:
        side = event.get("team_side") # 'home' or 'away'
        etype = event.get("event_type")
        
        if etype == "goal":
            if side == "home": home_score += 1
            elif side == "away": away_score += 1
        elif etype == "red_card":
            if side == "home": red_cards["home"] += 1
            elif side == "away": red_cards["away"] += 1
            
    # 2. Parse Stats (Live xG, etc.)
    live_xg = {"home": 0.0, "away": 0.0}
    stats_arr = stats_payload.get("data", {}).get("stats", [])
    
    if isinstance(stats_arr, list):
        for stat in stats_arr:
            stype = str(stat.get("type", "")).lower()
            if "xg" in stype or "expected goals" in stype:
                live_xg["home"] = float(stat.get("home", 0.0))
                live_xg["away"] = float(stat.get("away", 0.0))
                
    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_score": home_score,
        "away_score": away_score,
        "current_minute": 90 if match_info.get("status") == "complete" else "Live",
        "red_cards": red_cards,
        "live_xg": live_xg,
        "starting_xi": {"home": [], "away": []}
    }

if match_id_to_test:
    parsed_data = parse_footballdata_to_v4(stats_data, events_data)
    print("✅ Parsed V4 Model Input:\n")
    print(json.dumps(parsed_data, indent=2))
    
    print("\n" + "="*50 + "\n")
    print("🔍 Let's also inspect the raw 'stats' array to see exactly how expected goals are labeled:")
    if stats_data and "stats" in stats_data.get("data", {}):
        # Limit the output so we just see the types of stats available
        print(json.dumps(stats_data["data"]["stats"], indent=2)[:800])
    else:
        print("No 'stats' array found in stats_data.")